# Chess experiment — results & progress analysis

Thesis figures for the chess experiment: how infrastructure built **around a fixed
model** (Gemma-4-31B-it, temperature 1.0) moved its measured strength across four
development phases. Data: `analysis/data/games.csv` + `wiki_growth.csv`, built by
`analysis/build_dataset.py` (re-run it after new batches; it is incremental).
Provenance & narrative: `knowledge-base/work/experiment-chess-results-and-phases.md`.

Phases: **P1** minimal tools (legal moves + make_move) · **P2** visualization ·
**P3** mating & blunder-avoidance (user-directed, PRs 2-4) · **P4** the autonomous
fix loop (Claude-driven, PR 5). Ranked Elo milestones: ≤700 → 793.6 → 968.4 → 1311+.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import json
from pathlib import Path

DATA = Path("data")
df = pd.read_csv(DATA / "games.csv", parse_dates=["datetime"])
wiki = pd.read_csv(DATA / "wiki_growth.csv", parse_dates=["date"])

PHASES = ["P1-minimal-tools", "P2-visualization", "P3-mating-blunders", "P4-autonomous-loop"]
PHASE_COLORS = {"P1-minimal-tools": "#d9d9d9", "P2-visualization": "#c6dbef",
                "P3-mating-blunders": "#fdd0a2", "P4-autonomous-loop": "#c7e9c0"}
PR_MERGES = [("2026-05-26", "PR1 visualization"), ("2026-06-15", "PR2 mating"),
             ("2026-06-20", "PR3 conversion"), ("2026-06-23", "PR4 gates"),
             ("2026-07-03", "PR5 autonomous loop")]

ranked = df[(df.kind == "ranked") & (~df.aborted) & df.result.isin(["1-0","0-1","1/2-1/2"])].copy()
ranked = ranked.sort_values("datetime").reset_index(drop=True)
ranked["game_no"] = range(1, len(ranked) + 1)
full = df[(~df.is_puzzle_mode) & (~df.aborted) & df.result.isin(["1-0","0-1","1/2-1/2"])]
print(f"{len(df)} games total | {len(ranked)} ranked | phases:", df.phase.value_counts().to_dict())

## 1 — Ranked Elo trajectory (the headline figure)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
# phase bands
bounds = {}
for ph in PHASES:
    sub = ranked[ranked.phase == ph]
    if len(sub):
        bounds[ph] = (sub.game_no.min(), sub.game_no.max())
        ax.axvspan(sub.game_no.min() - 0.5, sub.game_no.max() + 0.5,
                   color=PHASE_COLORS[ph], alpha=0.5,
                   label=ph.split("-", 1)[1].replace("-", " "))
ax.plot(ranked.game_no, ranked.elo_after.astype(float), lw=1.8, color="#333")
ax.scatter(ranked.game_no, ranked.elo_after.astype(float), s=8, color="#333")
ax.set_xlabel("ranked game #"); ax.set_ylabel("agent Elo (post-game)")
ax.set_title("Agent Elo over ranked games — same model, four infrastructure phases")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig("figures/elo_trajectory.png", dpi=200); plt.show()

## 2 — Move quality by phase (accuracy & blunder rate)

In [ ]:
fq = full.dropna(subset=["accuracy"]).copy()
fq["blunders_per_100"] = 100 * fq.blunders / fq.plies.clip(lower=1) * 2  # per 100 own moves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, col, title in [(axes[0], "accuracy", "game accuracy % (agent)"),
                       (axes[1], "blunders_per_100", "blunders per 100 own moves")]:
    data = [fq[fq.phase == ph][col].dropna() for ph in PHASES]
    bp = ax.boxplot(data, tick_labels=[p.split("-")[0] for p in PHASES], patch_artist=True)
    for patch, ph in zip(bp["boxes"], PHASES):
        patch.set_facecolor(PHASE_COLORS[ph])
    ax.set_title(title); ax.grid(alpha=0.25)
    for i, d in enumerate(data, 1):
        if len(d): ax.annotate(f"n={len(d)}", (i, ax.get_ylim()[0]), ha="center", fontsize=8)
plt.suptitle("Move quality by development phase (full games, ranked + experimental)")
plt.tight_layout(); plt.savefig("figures/quality_by_phase.png", dpi=200); plt.show()

## 3 — The loss-profile shift: self-destruction → outplayed

Worst single-move win%-drop in each **lost** game. Early phases lose by one
catastrophic move; after the P4 loop the losses are gradual.

In [ ]:
losses = full[(full.result == "0-1") & full.worst_winpct_drop.notna()]
fig, ax = plt.subplots(figsize=(9, 4.5))
data = [losses[losses.phase == ph].worst_winpct_drop for ph in PHASES]
bp = ax.boxplot(data, tick_labels=[p.split("-")[0] for p in PHASES], patch_artist=True)
for patch, ph in zip(bp["boxes"], PHASES): patch.set_facecolor(PHASE_COLORS[ph])
ax.set_ylabel("worst single-move win% lost (per lost game)")
ax.set_title("How games are lost: one catastrophic move vs gradual outplay")
ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig("figures/loss_profile.png", dpi=200); plt.show()

## 4 — Knowledge component: wiki growth vs Elo

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(wiki.date, wiki.words, color="#2c7fb8", lw=2, label="wiki words")
ax1.set_ylabel("agent-wiki size (words)", color="#2c7fb8")
ax1.tick_params(axis="y", labelcolor="#2c7fb8")
ax2 = ax1.twinx()
ax2.plot(ranked.datetime, ranked.elo_after.astype(float), color="#333", lw=1.5, label="ranked Elo")
ax2.set_ylabel("ranked Elo", color="#333")
for d, lbl in PR_MERGES:
    ax1.axvline(pd.Timestamp(d), color="grey", ls=":", lw=1)
    ax1.annotate(lbl, (pd.Timestamp(d), ax1.get_ylim()[1]*0.97), rotation=90,
                 fontsize=7, va="top", color="grey")
ax1.set_title("The agent's knowledge wiki grows; the same model plays stronger")
plt.tight_layout(); plt.savefig("figures/wiki_vs_elo.png", dpi=200); plt.show()

## 5 — Skills component: tool usage per turn & tool mix by phase

In [ ]:
tm = full.dropna(subset=["tools_per_turn"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
data = [tm[tm.phase == ph].tools_per_turn for ph in PHASES]
bp = axes[0].boxplot(data, tick_labels=[p.split("-")[0] for p in PHASES], patch_artist=True)
for patch, ph in zip(bp["boxes"], PHASES): patch.set_facecolor(PHASE_COLORS[ph])
axes[0].set_title("tool calls per turn"); axes[0].grid(alpha=0.25)

# tool mix: aggregate tool_mix JSON per phase
mix = {}
for ph in PHASES:
    c = {}
    for s in full[full.phase == ph].tool_mix.dropna():
        for k, v in json.loads(s).items():
            c[k] = c.get(k, 0) + v
    mix[ph] = c
tools = sorted({t for c in mix.values() for t in c},
               key=lambda t: -sum(c.get(t, 0) for c in mix.values()))[:8]
bottoms = [0]*len(PHASES)
for t in tools:
    vals = []
    for i, ph in enumerate(PHASES):
        tot = sum(mix[ph].values()) or 1
        vals.append(100 * mix[ph].get(t, 0) / tot)
    axes[1].bar([p.split("-")[0] for p in PHASES], vals, bottom=bottoms, label=t.replace("chess__",""))
    bottoms = [b + v for b, v in zip(bottoms, vals)]
axes[1].set_title("tool mix (% of calls)"); axes[1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
plt.suptitle("Skills component: how the agent uses its tools, by phase")
plt.tight_layout(); plt.savefig("figures/tool_usage.png", dpi=200); plt.show()

## 6 — The P4 fix loop: per-batch class metrics

Numbers from the KB log (2026-07-02/03 entries); each 14-game experimental batch
validated one fix iteration. Aggregate blunder rate fell 52% while each
iteration's *targeted class* collapsed.

In [ ]:
loop = pd.DataFrame([
    ("pre-fix",              7.42, 16, "—"),
    ("iter1 proof-audit",    5.35,  7, "certified-unsound-sac overrides (Qxh7+ class eliminated)"),
    ("iter2 exchange-order", 5.80,  6, "ignored-pre-existing-hangs 5→1"),
    ("iter3 dangerous-checks",5.81, 6, "check-led loss moves 2/4→0/3"),
    ("iter4 PROMOTE",        4.83,  6, "promotion drills 17/20 (implicit baseline 0/20)"),
    ("iter5 anti-drift",     3.59,  5, "loss profile → gradual outplays"),
], columns=["batch", "blunder_rate_pct", "blunder_overrides", "targeted class result"])
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(loop))
ax.bar(x, loop.blunder_rate_pct, color="#fdae6b", label="all-move blunder rate %")
ax2 = ax.twinx()
ax2.plot(x, loop.blunder_overrides, "o-", color="#333", label="blunder-overrides / batch")
ax.set_xticks(x); ax.set_xticklabels(loop.batch, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("blunder rate %"); ax2.set_ylabel("blunder-overrides")
ax.set_title("The P4 autonomous fix loop: five iterations, five 14-game validation batches")
fig.legend(loc="upper right", bbox_to_anchor=(0.98, 0.92), fontsize=8)
plt.tight_layout(); plt.savefig("figures/fix_loop.png", dpi=200); plt.show()
loop

## 7 — Context management: prompt size per turn over time

In [ ]:
pc = full.dropna(subset=["prompt_chars_mean"]).sort_values("datetime")
fig, ax = plt.subplots(figsize=(11, 4))
for ph in PHASES:
    sub = pc[pc.phase == ph]
    ax.scatter(sub.datetime, sub.prompt_chars_mean, s=12, color=PHASE_COLORS[ph],
               edgecolors="#555", linewidths=0.3, label=ph.split("-")[0])
ax.set_ylabel("mean prompt length per turn (chars)")
ax.set_title("Context management: what the agent is shown each turn")
ax.legend(fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.savefig("figures/prompt_size.png", dpi=200); plt.show()

## 8 — Win rate by opponent level and phase

In [ ]:
wr = full[full.opponent.notna() & full.opponent.str.startswith("chesscom", na=False)].copy()
wr["opp"] = wr.opponent_elo.astype(float)
wr["score"] = wr.result.map({"1-0": 1.0, "0-1": 0.0, "1/2-1/2": 0.5})
tab = wr.pivot_table(index="opp", columns="phase", values="score", aggfunc=["mean", "count"])
tab